# 03 · Baseline TF-IDF + Ridge for Empathy (canonical)

Canonical Empathy-only baseline for the thesis. Following the
multi-target exploration in `02_baseline_multi_target.ipynb`, this
notebook focuses on a single target — Empathy — using TF-IDF features
combined with Ridge regression on the official WASSA CONV-Turn splits.
The trained model and evaluation results are saved as reference
artifacts used throughout the rest of the thesis.

**Configuration.** `TfidfVectorizer(ngram_range=(1,2), max_features=5000)`
with `Ridge(alpha=3.0)`. This is the Day-5 canonical configuration.
Subsequent hyperparameter sweeps (Days 8-10, notebook 04) refine each
of these choices; the final tuned configuration lives in
`04_tfidf_sweeps.ipynb`.


## 0 · Setup


In [1]:
!git clone https://github.com/DavorSopar/thesis-empathy.git /content/thesis-empathy
import sys
sys.path.insert(0, '/content/thesis-empathy/src')

Cloning into '/content/thesis-empathy'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 133 (delta 55), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 2.05 MiB | 5.35 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score

# Make src/ importable whether run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir():
    pass
elif (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from data import load_convt, impute_selfdisclosure, TARGETS

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

## 1 · Load the three official WASSA splits

The WASSA-released training file contains all conversations from the
underlying corpus, including those subsequently released as development
and test splits. We filter the training set to exclude any conversation
appearing in dev or test, yielding a conversation-disjoint partition.


In [3]:
# Load all three splits
train_raw = load_convt('train')
dev = load_convt('dev')
test = load_convt('test')

# Filter train: remove any conversation appearing in dev or test
excluded_ids = set(dev.conversation_id) | set(test.conversation_id)
train = train_raw[~train_raw.conversation_id.isin(excluded_ids)].reset_index(drop=True)

# Sanity check: no conversation overlap between splits
assert not (set(train.conversation_id) & set(dev.conversation_id))
assert not (set(train.conversation_id) & set(test.conversation_id))
assert not (set(dev.conversation_id) & set(test.conversation_id))

split_tbl = pd.DataFrame({
    'turns':         [len(train), len(dev), len(test)],
    'conversations': [train.conversation_id.nunique(),
                      dev.conversation_id.nunique(),
                      test.conversation_id.nunique()],
}, index=['train', 'dev', 'test'])
print(split_tbl)
print('\nAll three splits are conversation-disjoint.')


       turns  conversations
train   9330            405
dev      990             33
test    2061             63

All three splits are conversation-disjoint.


## 2 · Metric helpers


In [4]:
def pearson(y_true, y_pred):
    """Pearson r; returns NaN when either side is constant (undefined)."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def score_all(y_true, y_pred):
    return {
        "pearson": pearson(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

## 3 · TF-IDF features

The vectorizer is fit on the training split only, then applied to
development and test splits via `.transform()` (no re-fitting) to prevent
vocabulary leakage.


In [5]:
# Fit the vectorizer on TRAIN text only, then transform every split with it.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train = vectorizer.fit_transform(train["text"])
X_dev = vectorizer.transform(dev["text"])
X_test = vectorizer.transform(test["text"])
feature_names = np.array(vectorizer.get_feature_names_out())
print("TF-IDF matrix:", X_train.shape, "(train)")
print("Vocabulary size:", len(feature_names))

TF-IDF matrix: (9330, 5000) (train)
Vocabulary size: 5000


## 4 · Train Ridge and evaluate on all three splits

Train Ridge on the training split, then evaluate on train (to detect
overfitting), dev (for reference), and test (the number reported in the
thesis Results section).


In [6]:
BEST_ALPHA = 3.0   # from Day 3 sweep
TARGET = 'Empathy'

y_train_e = train[TARGET]
y_dev_e = dev[TARGET]
y_test_e = test[TARGET]

model = Ridge(alpha=BEST_ALPHA).fit(X_train, y_train_e)

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r = np.corrcoef(y_true, y_pred)[0, 1]
    print(f"{label:25s}  MAE={mae:.4f}  RMSE={rmse:.4f}  Pearson r={r:.4f}")
    return {'label': label, 'mae': mae, 'rmse': rmse, 'pearson': r}
results = [
    evaluate(y_train_e, model.predict(X_train), "Baseline Ridge · train"),
    evaluate(y_dev_e,   model.predict(X_dev),   "Baseline Ridge · dev"),
    evaluate(y_test_e,  model.predict(X_test),  "Baseline Ridge · test"),
]

Baseline Ridge · train     MAE=0.4811  RMSE=0.6050  Pearson r=0.7661
Baseline Ridge · dev       MAE=0.7640  RMSE=0.9253  Pearson r=0.5471
Baseline Ridge · test      MAE=0.9647  RMSE=1.1650  Pearson r=0.4941


## 5 · Persist model, vectorizer, and results

Saved to `results/` and `models/` so downstream notebooks can reload the
model without retraining, and so the thesis Results section can be
written from a canonical CSV.


In [7]:
import joblib

MODELS_DIR = REPO_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Save the results table
pd.DataFrame(results).to_csv(
    RESULTS_DIR / 'baseline_ridge_empathy_results.csv', index=False
)

# Save the trained Ridge model and its vectorizer
joblib.dump(model,      MODELS_DIR / 'baseline_ridge_empathy.pkl')
joblib.dump(vectorizer, MODELS_DIR / 'baseline_vectorizer_empathy.pkl')

print('Saved results/baseline_ridge_empathy_results.csv')
print('Saved models/baseline_ridge_empathy.pkl')
print('Saved models/baseline_vectorizer_empathy.pkl')


Saved results/baseline_ridge_empathy_results.csv
Saved models/baseline_ridge_empathy.pkl
Saved models/baseline_vectorizer_empathy.pkl
